# Jerárquico con Process.hierarchical

Clasificación: **Proceso jerárquico.** Un manager LLM coordina especialistas que pueden delegarse trabajo entre sí.

## Diferencia con el notebook 4 (orchestrator)

En el notebook 4 usamos `Process.hierarchical` con una sola task. El manager decidía a quién delegar esa task única.

Aquí vamos un paso más allá: el manager tiene múltiples tasks que supervisar, y los agentes tienen `allow_delegation=True` para que puedan redelegar entre ellos. El manager decide el orden, asigna, y puede pedir correcciones.

## Cómo funciona

```python
crew = Crew(
    agents=[director, logistica, experiencias, transporte],
    tasks=[task_logistica, task_experiencias, task_transporte, task_final],
    process=Process.hierarchical,
    manager_llm="gpt-4o-mini",
)
```

El manager LLM:
1. Lee todas las tasks pendientes
2. Decide qué agente ejecuta cada una
3. Puede re-asignar si un agente no da un resultado satisfactorio
4. Los agentes con `allow_delegation=True` pueden pedir ayuda a otros agentes del crew

In [ ]:
!uv pip install -r requirements.txt --quiet

In [ ]:
from dotenv import load_dotenv
import nest_asyncio

load_dotenv()
nest_asyncio.apply()

In [ ]:
from crewai import Agent, Task, Crew, Process

inputs = {"destino": "Islandia", "dias": 5, "personas": 2, "presupuesto": 2200}

# Agentes especializados con delegación habilitada
logistica = Agent(
    role="Coordinador de Logística",
    goal="Resolver vuelos y alojamiento dentro del presupuesto",
    backstory="Gestionas la parte logística de viajes: vuelos, transfers y alojamiento.",
    allow_delegation=True,
)

experiencias = Agent(
    role="Coordinador de Experiencias",
    goal="Diseñar actividades y experiencias para el destino",
    backstory="Conoces atracciones, restaurantes y experiencias locales. Puedes pedir ayuda al de transporte para coordinar desplazamientos.",
    allow_delegation=True,
)

transporte = Agent(
    role="Especialista en Transporte",
    goal="Proponer opciones de transporte entre puntos del viaje",
    backstory="Conoces bus, tren, taxi, coche de alquiler y ferry. Calculas tiempos y costes.",
    allow_delegation=False,
)

director = Agent(
    role="Director de Agencia",
    goal="Ensamblar un itinerario final coherente con todas las partes dentro del presupuesto",
    backstory="Recibes los reportes de logística, experiencias y transporte y los integras en un plan dia a dia.",
    allow_delegation=False,
)

# Tasks que el manager asignará
task_logistica = Task(
    description="Busca vuelos y alojamiento para {personas} personas, {dias} dias en {destino}. Presupuesto total: {presupuesto} EUR.",
    expected_output="Opciones de vuelo y alojamiento con precios.",
    agent=logistica,
)

task_experiencias = Task(
    description="Propón actividades para {dias} dias en {destino} para {personas} personas. Presupuesto total: {presupuesto} EUR.",
    expected_output="Plan de actividades dia por dia con coste estimado.",
    agent=experiencias,
)

task_transporte = Task(
    description="Propón opciones de transporte para moverse en {destino} durante {dias} dias. Presupuesto total: {presupuesto} EUR.",
    expected_output="Opciones de transporte con precio y tiempo.",
    agent=transporte,
)

task_final = Task(
    description="Con los reportes anteriores, ensambla un itinerario dia a dia dentro de {presupuesto} EUR para {personas} personas en {destino}.",
    expected_output="Itinerario final con vuelos, alojamiento, actividades, transporte, coste por partida y total.",
    agent=director,
    context=[task_logistica, task_experiencias, task_transporte],
)

crew = Crew(
    agents=[logistica, experiencias, transporte, director],
    tasks=[task_logistica, task_experiencias, task_transporte, task_final],
    process=Process.hierarchical,
    manager_llm="gpt-4o-mini",
    verbose=True,
)

result = await crew.kickoff_async(inputs=inputs)
print(result.raw)

## Qué define este patrón

El manager LLM supervisa múltiples tasks y las asigna a los agentes. Los agentes con `allow_delegation=True` (logística y experiencias) pueden pedir ayuda a otros agentes del crew. Transporte y director no delegan, solo ejecutan.

Diferencia con el notebook 4: allí había una sola task general y el manager decidía a quién consultar. Aquí hay 4 tasks explícitas que el manager orquesta, y los agentes pueden colaborar entre sí (experiencias puede pedirle a transporte que calcule tiempos para coordinar actividades).